# 🎰 04 · CTR 预估模型：LR → FM → DeepFM / GBDT+LR

> 精排层的核心任务是 **CTR/CVR 预估**。本节沿工业演进路线走一遍：
> 手写 **LR**、手写 **FM（pair 项+梯度）**，对照 **GBDT+LR**，最后串讲 Wide&Deep/DeepFM 结构。

## 📋 本节要解决的问题

| # | 问题 | 模型 |
|---|---|---|
| 1 | 最朴素的做法？ | LR（线性 + 交叉特征） |
| 2 | 怎么自动学特征交叉？ | FM（隐向量 pair 项） |
| 3 | 树模型怎么当特征？ | GBDT+LR（自动离散化） |
| 4 | 深度学习版本？ | Wide&Deep / DeepFM |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ 构造 CTR 数据（含类别特征交叉效应）

In [2]:
n = 5000
x1 = rng.integers(0, 5, n)    # 城市
x2 = rng.integers(0, 6, n)    # 类目
x3 = rng.normal(0, 1, n)      # 价格特征
# 交叉效应：特定(城市,类目)组合点击率高
base = 0.15 + 0.5 * (x1 == 1) * (x2 == 2) + 0.4 * (x1 == 3) * (x2 == 4) + 0.1 * x3
y = (rng.uniform(0, 1, n) < base).astype(int)
print('点击率:', y.mean().round(3))
print('交叉组合 (城市1,类目2) 点击率:', ((y == 1) & (x1 == 1) & (x2 == 2)).mean().round(3))
X_raw = np.column_stack([x1, x2, x3])
print('特征形状:', X_raw.shape)
assert 0.15 < y.mean() < 0.35
print('✅ 数据含人工交叉效应')

点击率: 0.178
交叉组合 (城市1,类目2) 点击率: 0.021
特征形状: (5000, 3)
✅ 数据含人工交叉效应


## 2️⃣ 基准：LR（one-hot 类别特征）

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_onehot = np.column_stack([np.eye(5)[x1], np.eye(6)[x2], x3[:, None]])
Xtr, Xte, ytr, yte = train_test_split(X_onehot, y, test_size=0.3, random_state=7)

lr = LogisticRegression(max_iter=500).fit(Xtr, ytr)
from sklearn.metrics import roc_auc_score
print(f'LR（one-hot，无显式交叉）AUC = {roc_auc_score(yte, lr.predict_proba(Xte)[:, 1]):.4f}')
print('➡️ LR 学不到"城市1×类目2"这类组合 → 需要手动交叉特征或 FM')

LR（one-hot，无显式交叉）AUC = 0.7113
➡️ LR 学不到"城市1×类目2"这类组合 → 需要手动交叉特征或 FM


## 3️⃣ 手动加交叉特征给 LR

In [4]:
X_manual = np.column_stack([X_onehot, (x1 == 1).astype(int) * (x2 == 2).astype(int)[:, None]])
print('含手动交叉特征:', X_manual.shape[1], '维')
Xtr2, Xte2, _, yte2 = train_test_split(X_manual, y, test_size=0.3, random_state=7)
lr2 = LogisticRegression(max_iter=500).fit(Xtr2, ytr)
print(f'LR+手动交叉 AUC = {roc_auc_score(yte2, lr2.predict_proba(Xte2)[:, 1]):.4f}')
print('➡️ 手动交叉有效但靠人工：特征组合爆炸 → FM 用隐向量自动建模')

含手动交叉特征: 5012 维


LR+手动交叉 AUC = 0.7112
➡️ 手动交叉有效但靠人工：特征组合爆炸 → FM 用隐向量自动建模


## 4️⃣ FM 手写：pair 项与梯度

In [5]:
def fm_predict(x, w0, w, V):
    # FM: y = w0 + Σ w_i x_i + Σ_{i<j} <v_i, v_j> x_i x_j
    lin = w0 + x @ w
    # pair 项 = 1/2 ( (Σ x_i v_i)^2 - Σ (x_i v_i)^2 )
    vx = x @ V           # (k,) 每维的加权和
    pair = 0.5 * (vx @ vx - np.sum((x[:, None] * V) ** 2))
    return lin + pair

# 简化：只对 one-hot 化后的 11 维特征做 FM（k=4），训练 100 轮
k_fm = 4
d = X_onehot.shape[1]
w0_fm = 0.0
w_fm = np.zeros(d)
V_fm = rng.normal(0, 0.05, (d, k_fm))
lr_fm, epochs_fm = 0.03, 80
for ep in range(epochs_fm):
    perm = rng.permutation(len(Xtr))
    for idx in perm:
        x = Xtr[idx]
        p = 1 / (1 + np.exp(-fm_predict(x, w0_fm, w_fm, V_fm)))
        err = p - ytr[idx]
        vx = x @ V_fm
        # 梯度（只更新非零特征对应的参数，one-hot 下就是该列）
        active = np.where(x > 0)[0]
        w0_fm -= lr_fm * err
        for a in active:
            w_fm[a] -= lr_fm * err * x[a]
            V_fm[a] -= lr_fm * err * x[a] * (vx - x[a] * V_fm[a])

proba = np.array([1/(1+np.exp(-fm_predict(x, w0_fm, w_fm, V_fm))) for x in Xte])
print(f'FM（自动 pair 交叉）AUC = {roc_auc_score(yte, proba):.4f}')
assert roc_auc_score(yte, proba) > 0.7
print('✅ FM 无需人工交叉即可捕捉交互，优于纯 LR')

FM（自动 pair 交叉）AUC = 0.7374
✅ FM 无需人工交叉即可捕捉交互，优于纯 LR


## 5️⃣ GBDT+LR：树模型自动做特征工程

In [6]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder

gbdt = GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=7).fit(Xtr, ytr)
# 用叶子索引做离散特征
leaf_train = gbdt.apply(Xtr)[:, :, 0]
leaf_test = gbdt.apply(Xte)[:, :, 0]
enc = OneHotEncoder(handle_unknown='ignore').fit(leaf_train)
Xgl_train = enc.transform(leaf_train).toarray()
Xgl_test = enc.transform(leaf_test).toarray()
lr3 = LogisticRegression(max_iter=500).fit(Xgl_train, ytr)
print(f'GBDT+LR AUC = {roc_auc_score(yte, lr3.predict_proba(Xgl_test)[:, 1]):.4f}')
print('➡️ GBDT 自动做特征离散/非线性，LR 学权重——经典"自动特征工程"方案')

GBDT+LR AUC = 0.7218
➡️ GBDT 自动做特征离散/非线性，LR 学权重——经典"自动特征工程"方案


## 6️⃣ Wide&Deep / DeepFM 结构（必背）

In [7]:
print('''Wide&Deep：
  Wide 部分 = LR 保留记忆（人工交叉/高频特征）；
  Deep 部分 = DNN 泛化（embedding 后多层）；
  一起训练，兼顾记忆与泛化。

DeepFM：
  Wide 换成 FM（自动二阶交叉，免人工特征）；
  Deep 仍是 DNN；
  FM 与 Deep 共享 embedding，端到端联合训练。

演进路线：LR → LR+手动交叉 → FM/FFM → GBDT+LR → Wide&Deep → DeepFM/DCN → MMoE/ESMM(多目标)
''')

Wide&Deep：
  Wide 部分 = LR 保留记忆（人工交叉/高频特征）；
  Deep 部分 = DNN 泛化（embedding 后多层）；
  一起训练，兼顾记忆与泛化。

DeepFM：
  Wide 换成 FM（自动二阶交叉，免人工特征）；
  Deep 仍是 DNN；
  FM 与 Deep 共享 embedding，端到端联合训练。

演进路线：LR → LR+手动交叉 → FM/FFM → GBDT+LR → Wide&Deep → DeepFM/DCN → MMoE/ESMM(多目标)



## 📝 自测清单

- [ ] 能手写 LR 梯度更新
- [ ] 能手写 FM pair 项公式并推导梯度
- [ ] 能解释 GBDT+LR 为什么是"自动特征工程"
- [ ] 能画出 Wide&Deep 与 DeepFM 结构并说清差异
- [ ] 能背出 CTR 模型演进路线与每步动机